# Profiling dan EDA Dataset Geohazard Newswatch

Notebook ini membangun profil lengkap dari folder `data/`:
- jumlah artikel pada tahap pengumpulan, konsolidasi, QC, cleaning/deduplication, dan output akhir;
- distribusi delapan kategori bencana;
- distribusi portal berita;
- distribusi waktu publikasi;
- distribusi bahasa;
- pemeriksaan duplikasi, kualitas teks, URL, tanggal, dan missing values;
- profil gold set berlabel.

Semua angka dibaca langsung dari artefak CSV/JSON aktual.

In [ ]:
from pathlib import Path
import json
import ast
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 6)

ROOT = Path(r"D:\MAGANGHUB BATCH 3\Scraper\geolocation3")
DATA = ROOT / "data"

def read_csv(relative):
    return pd.read_csv(DATA / relative, encoding="utf-8-sig")

def parse_query_groups(value):
    if pd.isna(value):
        return []
    try:
        parsed = json.loads(str(value))
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed]
    except Exception:
        pass
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, (list, tuple)):
            return [str(v).strip() for v in parsed]
    except Exception:
        pass
    return [str(value).strip()] if str(value).strip() else []

HAZARDS = [
    "gunungapi", "gempabumi", "tsunami", "tanah_longsor",
    "likuefaksi", "abrasi_pantai", "penurunan_tanah",
    "semburan_gas_lumpur",
]

raw = {}
for path in sorted((DATA / "raw").glob("*.csv")):
    raw[path.name] = pd.read_csv(path, encoding="utf-8-sig")

newswatch_long = read_csv("interim/consolidation/newswatch_long.csv")
articles = read_csv("interim/consolidation/articles.csv")
preprocessed = read_csv("interim/quality_control/newswatch_preprocessed.csv")
qc_issues = read_csv("interim/quality_control/qc_issues.csv")
clean = read_csv("processed/articles_clean.csv")
indonesia = read_csv("processed/articles_indonesia.csv")
prompt = read_csv("processed/articles_prompt.csv")
goldset = read_csv("interim/goldset/goldset_labeled.csv")

print("Paket dan dataset berhasil dimuat.")

## 1. Ringkasan Funnel Pengumpulan dan Pemrosesan

In [ ]:
stage_counts = pd.DataFrame([
    ("Raw CSV - baris mentah", sum(len(df) for df in raw.values())),
    ("Newswatch - baris panjang", len(newswatch_long)),
    ("Artikel unik dari URL", len(articles)),
    ("Setelah QC/preprocessing", len(preprocessed)),
    ("Setelah cleaning/dedup", len(clean)),
    ("Hanya bahasa Indonesia/und", len(indonesia)),
    ("Siap prompt (duplikasi tepat dihapus)", len(prompt)),
    ("Gold set berlabel", len(goldset)),
], columns=["Tahap", "Jumlah"])

display(stage_counts)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=stage_counts, y="Tahap", x="Jumlah", ax=ax, palette="Blues_r")
ax.set_title("Funnel Jumlah Artikel per Tahap")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
plt.tight_layout()
plt.show()

## 2. Profil Pengumpulan Raw

In [ ]:
raw_counts = pd.DataFrame({
    "sourcefile": list(raw.keys()),
    "rows": [len(df) for df in raw.values()],
})
raw_counts["hazard"] = raw_counts["sourcefile"].str.replace(r"^geohazard_\d+_", "", regex=True).str.replace(r"\d+$", "", regex=True).str.replace(r"\.csv$", "", regex=True)
raw_counts = raw_counts.sort_values("rows", ascending=False)

display(raw_counts)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=raw_counts, x="hazard", y="rows", ax=ax, palette="mako")
ax.set_title("Distribusi Baris Mentah berdasarkan Kelompok Bencana")
ax.set_xlabel("Kelompok bencana")
ax.set_ylabel("Jumlah baris")
ax.tick_params(axis="x", rotation=35)
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
plt.tight_layout()
plt.show()

with open(DATA / "interim/consolidation/validation_report.json", encoding="utf-8") as f:
    validation_report = json.load(f)
pd.DataFrame([{
    "Rows sebelum concat": validation_report["rows_before_concat"],
    "Rows setelah concat": validation_report["rows_after_concat"],
    "Artikel unik setelah agregasi": validation_report["rows_after_article_aggregation"],
    "Rows teragregasi": validation_report["rows_collapsed_by_article_id"],
    "Artikel multi-kueri": validation_report["articles_in_multiple_query_groups"],
    "Grup konten duplikat lintas URL": validation_report["content_duplicate_groups_with_different_urls"],
}]).T.rename(columns={0:"Nilai"})

## 3. Distribusi Jenis Bencana

In [ ]:
article_groups = articles["query_group"].apply(parse_query_groups)
hazard_counts = pd.Series(
    [hazard for groups in article_groups for hazard in groups],
    name="bencana"
).value_counts().reindex(HAZARDS).fillna(0).astype(int)
hazard_df = hazard_counts.rename_axis("jenis_bencana").reset_index(name="jumlah")
hazard_df["persen"] = 100 * hazard_df["jumlah"] / len(articles)
display(hazard_df)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=hazard_df, x="jenis_bencana", y="jumlah", ax=ax, palette="viridis")
ax.set_title("Distribusi Kelas Bencana pada Artikel Unik")
ax.set_xlabel("Jenis bencana")
ax.set_ylabel("Jumlah artikel")
ax.tick_params(axis="x", rotation=35)
for i, row in hazard_df.iterrows():
    ax.text(i, row["jumlah"], f"{row['jumlah']}\n({row['persen']:.1f}%)",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

multi_label_count = article_groups.apply(len).gt(1).sum()
print(f"Artikel unik dengan multi-label kueri: {multi_label_count} "
      f"({100*multi_label_count/len(articles):.2f}%)")

## 4. Distribusi Portal Berita

In [ ]:
source_frames = {}
for name, df in [
    ("raw", newswatch_long),
    ("artikel_unik", articles),
    ("preprocessed", preprocessed),
    ("clean", clean),
    ("indonesia", indonesia),
    ("prompt", prompt),
]:
    col = "source"
    source_frames[name] = df[col].fillna("(kosong)").value_counts().rename(name)

source_compare = pd.concat(source_frames.values(), axis=1).fillna(0).astype(int)
source_compare["total_tahap"] = source_compare.sum(axis=1)
display(source_compare.sort_values("prompt", ascending=False).head(30))

top = source_compare.sort_values("prompt", ascending=False).head(15)
top[["raw","artikel_unik","preprocessed","clean","indonesia","prompt"]].plot(
    kind="bar", figsize=(14, 7), width=0.82
)
plt.title("Distribusi Portal Berita pada 15 Portal Teratas")
plt.xlabel("Portal berita")
plt.ylabel("Jumlah artikel")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

portal_pie = clean["source"].fillna("(kosong)").value_counts().head(10)
portal_pie.plot.pie(autopct="%1.1f%%", figsize=(8, 8), startangle=90)
plt.title("Proporsi 10 Portal Teratas pada Data Bersih")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 5. Distribusi Waktu

In [ ]:
date_col = "publish_date_normalized" if "publish_date_normalized" in clean.columns else "publish_date"
clean_dates = pd.to_datetime(clean[date_col], errors="coerce")
monthly = clean_dates.dt.to_period("M").astype(str).value_counts().sort_index()
monthly_df = monthly.rename_axis("bulan").reset_index(name="jumlah")
display(monthly_df)

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(monthly_df["bulan"], monthly_df["jumlah"], marker="o", linewidth=2)
ax.set_title("Jumlah Artikel per Bulan")
ax.set_xlabel("Bulan")
ax.set_ylabel("Jumlah artikel")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

yearly = clean_dates.dt.year.value_counts().sort_index()
yearly_df = yearly.rename_axis("tahun").reset_index(name="jumlah")
display(yearly_df)

print("Rentang tanggal:",
      clean_dates.min().strftime("%Y-%m-%d") if clean_dates.notna().any() else None,
      "hingga",
      clean_dates.max().strftime("%Y-%m-%d") if clean_dates.notna().any() else None)

## 6. Distribusi Bahasa

In [ ]:
language_col = "language_code" if "language_code" in clean.columns else None
if language_col:
    lang = clean[language_col].fillna("(kosong)").value_counts().rename_axis("bahasa").reset_index(name="jumlah")
    lang["persen"] = 100 * lang["jumlah"] / len(clean)
    display(lang)

    lang.set_index("bahasa")["jumlah"].plot.pie(
        autopct="%1.1f%%", figsize=(7, 7), startangle=90
    )
    plt.title("Distribusi Bahasa Artikel")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

if "language_confidence" in clean.columns:
    display(clean["language_confidence"].describe().to_frame("confidence"))
    sns.histplot(clean["language_confidence"].dropna(), bins=20)
    plt.title("Distribusi Confidence Deteksi Bahasa")
    plt.xlabel("Confidence")
    plt.ylabel("Jumlah artikel")
    plt.tight_layout()
    plt.show()

## 7. Profil QC dan Panjang Teks

In [ ]:
qc_status = preprocessed["qc_status"].value_counts().rename_axis("status").reset_index(name="jumlah")
display(qc_status)

qc_status.set_index("status")["jumlah"].plot.pie(
    autopct="%1.1f%%", figsize=(6, 6), startangle=90
)
plt.title("Status QC")
plt.ylabel("")
plt.tight_layout()
plt.show()

length_cols = [c for c in ["title_length", "content_length", "content_word_count"]
               if c in preprocessed.columns]
profiles = preprocessed[length_cols].describe(percentiles=[.25,.5,.75,.95]).T
display(profiles)

if "content_length" in preprocessed.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(preprocessed["content_length"], bins=50, ax=axes[0])
    axes[0].set_title("Distribusi Panjang Konten")
    axes[0].set_xlabel("Karakter")
    sns.boxplot(x=preprocessed["content_length"], ax=axes[1])
    axes[1].set_title("Boxplot Panjang Konten")
    axes[1].set_xlabel("Karakter")
    plt.tight_layout()
    plt.show()

missing = preprocessed.isna().sum()
missing = missing[missing.gt(0)].sort_values(ascending=False)
missing_df = (missing.rename_axis("kolom").reset_index(name="missing"))
missing_df["persen"] = 100 * missing_df["missing"] / len(preprocessed)
display(missing_df)

if "qc_flags" in preprocessed.columns:
    flags = (preprocessed["qc_flags"].fillna("")
             .str.split(";").explode().str.strip())
    flags = flags[flags.ne("")]
    display(flags.value_counts().rename_axis("qc_flag").reset_index(name="jumlah").head(30))

## 8. Perbandingan Sebelum dan Sesudah Pembersihan Teks

In [ ]:
compare = pd.DataFrame({
    "Tahap": ["Raw/consolidasi", "QC/preprocessed", "Clean/dedup"],
    "Jumlah artikel": [len(articles), len(preprocessed), len(clean)],
})
display(compare)

raw_lengths = newswatch_long["content"].fillna("").astype(str).str.len()
qc_lengths = preprocessed["content_cleaned"].fillna("").astype(str).str.len()
clean_lengths = clean["content_clean"].fillna("").astype(str).str.len() if "content_clean" in clean.columns else pd.Series(dtype=float)

length_compare = pd.DataFrame({
    "raw": raw_lengths,
}).describe()
length_compare["qc"] = qc_lengths.describe()
if len(clean_lengths):
    length_compare["clean"] = clean_lengths.describe()
display(length_compare)

to_plot = pd.DataFrame({
    "raw": raw_lengths,
    "qc": qc_lengths,
    "clean": clean_lengths if len(clean_lengths) else pd.Series(dtype=float),
})
to_plot.plot(kind="box", figsize=(10, 6), showfliers=False)
plt.title("Perbandingan Panjang Konten (Outlier Disembunyikan)")
plt.ylabel("Karakter")
plt.tight_layout()
plt.show()

## 9. Profil Duplikasi dan Near-Duplicate

In [ ]:
with open(DATA / "interim/deduplication/quality_report.json", encoding="utf-8") as f:
    dedup_report = json.load(f)

dedup_metrics = dedup_report["metrics"]
display(pd.DataFrame([
    ("Artikel input dedup", dedup_metrics["input_articles"]),
    ("Artikel dengan duplicate_of", dedup_metrics["articles_with_duplicate_of"]),
    ("Relasi duplikat tepat", dedup_metrics["exact_duplicate_relations"]),
    ("Pasangan kandidat near-duplikat", dedup_metrics["near_duplicate_candidate_pairs"]),
    ("Relasi near-duplikat ditetapkan", dedup_metrics["near_duplicate_relations_assigned"]),
    ("Artikel prompt", dedup_metrics["articles_in_prompt_output"]),
], columns=["Metrik", "Nilai"]))

if "duplicate_type" in clean.columns:
    dup_type = clean["duplicate_type"].fillna("(bukan duplikat)").value_counts()
    display(dup_type.rename_axis("tipe_duplikat").reset_index(name="jumlah"))

duplicate_pairs = read_csv("interim/deduplication/duplicate_pairs.csv")
near_review = read_csv("interim/deduplication/near_duplicate_manual_review.csv")
print("Jumlah pasangan duplicate_pairs:", len(duplicate_pairs))
print("Jumlah baris near_duplicate manual review:", len(near_review))
display(duplicate_pairs.head(10))
display(near_review.head(10))

## 10. Profil Gold Set

In [ ]:
gold_hazard_counts = pd.DataFrame({
    "jenis_bencana": HAZARDS,
    "jumlah": [goldset[c].fillna("").eq("yes").sum() for c in HAZARDS],
})
gold_hazard_counts["persen"] = 100 * gold_hazard_counts["jumlah"] / len(goldset)
display(gold_hazard_counts)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=gold_hazard_counts, x="jenis_bencana", y="jumlah", ax=ax, palette="rocket")
ax.set_title("Distribusi Label Bencana pada Gold Set")
ax.tick_params(axis="x", rotation=35)
for i, row in gold_hazard_counts.iterrows():
    ax.text(i, row["jumlah"], f"{row['jumlah']} ({row['persen']:.1f}%)",
            ha="center", va="bottom")
plt.tight_layout()
plt.show()

gold_relevant = goldset["relevant"].fillna("").value_counts().rename_axis("relevant").reset_index(name="jumlah")
display(gold_relevant)

uncertainty = goldset["goldset_uncertainty"].fillna("(kosong)").value_counts().rename_axis("ketidakpastian").reset_index(name="jumlah")
display(uncertainty)

event_count = goldset["event_report_count"].fillna("(kosong)").astype(str).value_counts().rename_axis("perkiraan_event").reset_index(name="jumlah")
display(event_count)

print("Validasi bukti kutipan:",
      int(goldset.loc[goldset["relevant"].eq("yes"), "evidence_text"].fillna("").ne("").sum()),
      "dari",
      int(goldset["relevant"].eq("yes").sum()),
      "artikel relevan")

## 11. Analisis Silang

In [ ]:
# Keterkaitan jenis bencana dengan portal teratas
clean_hazard = pd.DataFrame({
    "source": clean["source"].fillna("(kosong)"),
})
for i, hazard in enumerate(HAZARDS):
    pass

# Query_group diluaskan menjadi satu baris per artikel-kelas
cross = clean[["article_id","source","query_group","publish_date_normalized"]].copy()
cross["hazard"] = cross["query_group"].apply(parse_query_groups)
cross = cross.explode("hazard")
cross = cross[cross["hazard"].isin(HAZARDS)]

cross_table = pd.crosstab(cross["source"], cross["hazard"])
top_sources = cross["source"].value_counts().head(12).index
cross_top = cross_table.loc[top_sources]
display(cross_top)

sns.heatmap(cross_top, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Jenis Bencana vs Portal (12 Portal Teratas)")
plt.xlabel("Jenis bencana")
plt.ylabel("Portal")
plt.tight_layout()
plt.show()

# Ringkasan kalender untuk Indonesia
ind_dates = pd.to_datetime(indonesia["publish_date_normalized"], errors="coerce")
display(pd.DataFrame({
    "rentang_awal": [ind_dates.min()],
    "rentang_akhir": [ind_dates.max()],
    "artikel_indonesia": [len(indonesia)],
}).T.rename(columns={0:"Nilai"}))

## 12. Pemeriksaan Integritas dan Keluaran Ringkas

In [ ]:
integrity = pd.DataFrame([
    ("Raw baris", sum(len(df) for df in raw.values())),
    ("Artikel unik URL", articles["article_id"].nunique()),
    ("Duplikat article_id di artikel unik", articles["article_id"].duplicated().sum()),
    ("URL kosong pada artikel unik", articles["url_normalized"].fillna("").eq("").sum()),
    ("Content kosong pada artikel unik", articles["content"].fillna("").eq("").sum()),
    ("QC pass", (preprocessed["qc_status"] == "pass").sum()),
    ("QC non-pass", (preprocessed["qc_status"] != "pass").sum()),
    ("Bahasa id", (clean["language_code"] == "id").sum() if "language_code" in clean else np.nan),
    ("Bahasa en", (clean["language_code"] == "en").sum() if "language_code" in clean else np.nan),
    ("Bahasa und", (clean["language_code"] == "und").sum() if "language_code" in clean else np.nan),
    ("Gold set rows", len(goldset)),
], columns=["Pemeriksaan", "Nilai"])
display(integrity)

(DATA / "reports").mkdir(exist_ok=True)
stage_counts.to_csv(DATA / "reports/profile_stage_counts.csv", index=False)
hazard_df.to_csv(DATA / "reports/profile_hazard_distribution.csv", index=False)
source_compare.sort_values("prompt", ascending=False).to_csv(DATA / "reports/profile_source_distribution.csv")
monthly_df.to_csv(DATA / "reports/profile_monthly_distribution.csv", index=False)
gold_hazard_counts.to_csv(DATA / "reports/profile_goldset_distribution.csv", index=False)
integrity.to_csv(DATA / "reports/profile_integrity.csv", index=False)

print("Ringkasan CSV disimpan ke:", DATA / "reports")